In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')


INPUT_FILE = '/kaggle/input/datasets/jpcnvn/tensilde-data/tensile_data_with_strain(true data).xlsx'


OUTPUT_DIR = '/kaggle/working/'


L0 = 25.0       # gauge length (mm)
R  = 8.314      # gas constant J/(mol·K)
POLY_ORDER = 6  # polynomial order for SCAM

# SECTION 1: DATA LOADING & TRUE STRESS-STRAIN COMPUTATION

print("=" * 70)
print("SECTION 1: Loading data & computing true stress-strain")
print("=" * 70)

conditions = {
    'RT-0_001': (25, 0.001), 'RT-0_01': (25, 0.01), 'RT-0_1': (25, 0.1),
    '150-0_001': (150, 0.001), '150-0_01': (150, 0.01), '150-0_1': (150, 0.1),
    '250-0_001': (250, 0.001), '250-0_01': (250, 0.01), '250-0_1': (250, 0.1),
    '300-0_001': (300, 0.001), '300-0_01': (300, 0.01), '300-0_1': (300, 0.1),
    '350-0_001': (350, 0.001), '350-0_01': (350, 0.01), '350-0_1': (350, 0.1),
    '400-0_001': (400, 0.001), '400-0_01': (400, 0.01), '400-0_1': (400, 0.1),
    '450-0_001': (450, 0.001), '450-0_01': (450, 0.01), '450-0_1': (450, 0.1),
}

def get_sheet_name(T, sr):
    """Convert (T_celsius, strain_rate) to sheet name."""
    if T == 25:
        if sr == 0.001: return 'RT-0_001'
        elif sr == 0.01: return 'RT-0_01'
        elif sr == 0.1: return 'RT-0_1'
    if sr == 0.001: return f"{T}-0_001"
    elif sr == 0.01: return f"{T}-0_01"
    elif sr == 0.1: return f"{T}-0_1"

all_data = {}

for sheet_name, (T_C, sr) in conditions.items():
    df = pd.read_excel(INPUT_FILE, sheet_name=sheet_name)
    T_K = T_C + 273.15
    
    # Engineering strain from crosshead displacement (extensometer unreliable)
    eps_eng = df['Displacement'].values / L0
    # Engineering stress (Stress column = engineering stress in MPa)
    sigma_eng = df['Stress'].values
    
    # Convert to true stress -0_01"
    elif sr == 0.1: return f"{T}-0_1"

all_data = {}

for sheet_name, (T_C, sr) in conditions.items():
    df = pd.read_excel(INPUT_FILE, sheet_name=sheet_name)
    T_K = T_C + 273.15
    
    # Engineering strain from crosshead displacement (extensometer unreliable)
    eps_eng = df['Displacement'].values / L0
    # Engineering stress (Stress column = engineering stress in MPa)
    sigma_eng = df['Stress'].values
    
    # Convert to true stress and true strain
    sigma_true = sigma_eng * (1 + eps_eng)
    eps_true = np.log(1 + eps_eng)
    
    # Filter: skip elastic region + noise
    mask = (eps_eng > 0.001) & (sigma_true > 0)
    sigma_true = sigma_true[mask]
    eps_true = eps_true[mask]
    
    # Cut off after peak stress (UTS) — keep only up to peak
    peak_idx = np.argmax(sigma_true)
    sigma_true = sigma_true[:peak_idx + 1]
    eps_true = eps_true[:peak_idx + 1]
    
    all_data[sheet_name] = {
        'T_C': T_C, 'T_K': T_K, 'sr': sr,
        'eps_true': eps_true, 'sigma_true': sigma_true
    }
    print(f"  {sheet_name:>12s}: T={T_C:>4d}°C, ε̇={sr:.3f} s⁻¹, "
          f"pts={len(eps_true):>5d}, "
          f"ε=[{eps_true.min():.4f}, {eps_true.max():.4f}], "
          f"σ=[{sigma_true.min():.1f}, {sigma_true.max():.1f}] MPa")

print(f"\nTotal conditions loaded: {len(all_data)}")

# SECTION 2: SCAM PARAMETER FITTING AT EACH STRAIN LEVEL

print("\n" + "=" * 70)
print("SECTION 2: SCAM — Fitting α, n, Q, ln(A) at each strain level")
print("=" * 70)

# Only use 250–450°C (15 conditions) for Arrhenius fitting
# RT & 150°C excluded: DSA anomaly violates Arrhenius assumptions
scam_temps = [250, 300, 350, 400, 450]
scam_srs = [0.001, 0.01, 0.1]
T_K_arr = np.array([T + 273.15 for T in scam_temps])
ln_sr_arr = np.array([np.log(sr) for sr in scam_srs])

# Find common strain range across all 15 SCAM conditions
eps_max_common = min(
    all_data[get_sheet_name(T, sr)]['eps_true'].max()
    for T in scam_temps for sr in scam_srs
)
print(f"Common max strain (15 SCAM conditions): {eps_max_common:.4f}")

# Strain grid: 0.05 to eps_max_common, step 0.01
strain_levels = np.arange(0.05, eps_max_common - 0.005, 0.01)
print(f"Strain levels: {len(strain_levels)} points, "
      f"range [{strain_levels[0]:.3f}, {strain_levels[-1]:.3f}]")

# Interpolate stress at each strain level for each condition
stress_at_strain = {}
for T in scam_temps:
    for sr in scam_srs:
        sn = get_sheet_name(T, sr)
        d = all_data[sn]
        stress_at_strain[(T, sr)] = np.interp(
            strain_levels, d['eps_true'], d['sigma_true']
        )

# Fit 4 parameters at each strain level
results = {'strain': [], 'alpha': [], 'n': [], 'Q': [], 'lnA': []}

for i, eps in enumerate(strain_levels):
    # Stress matrix: shape (5 temps, 3 strain rates)
    sigma_matrix = np.array([
        [stress_at_strain[(T, sr)][i] for sr in scam_srs]
        for T in scam_temps
    ])
    
    # (a) ln(σ) vs ln(ε̇) at each T → slope = 1/n₁ (low stress approx.)
    n1_list = []
    for j in range(len(scam_temps)):
        slope = np.polyfit(ln_sr_arr, np.log(sigma_matrix[j, :]), 1)[0]
        n1_list.append(1.0 / slope)
    n1 = np.mean(n1_list)
    
    # (b) σ vs ln(ε̇) at each T → slope = 1/β (high stress approx.)
    beta_list = []
    for j in range(len(scam_temps)):
        slope = np.polyfit(ln_sr_arr, sigma_matrix[j, :], 1)[0]
        beta_list.append(1.0 / slope)
    beta = np.mean(beta_list)
    
    # (c) α = β / n₁
    alpha = abs(beta / n1)
    
    # (d) ln[sinh(ασ)] vs ln(ε̇) at each T → slope = 1/n
    n_list = []
    for j in range(len(scam_temps)):
        y = np.log(np.sinh(alpha * sigma_matrix[j, :]))
        slope = np.polyfit(ln_sr_arr, y, 1)[0]
        n_list.append(1.0 / slope)
    n_val = np.mean(n_list)
    
    # (e) ln[sinh(ασ)] vs 1/T at each ε̇ → slope = Q/(nR)
    S_list = []
    for k in range(len(scam_srs)):
        y = np.log(np.sinh(alpha * sigma_matrix[:, k]))
        x = 1.0 / T_K_arr
        slope = np.polyfit(x, y, 1)[0]
        S_list.append(slope)
    S = np.mean(S_list)
    Q = abs(S * n_val * R)
    
    # (f) ln(Z) vs ln[sinh(ασ)] → intercept = ln(A)
    lnZ_all, lnsinh_all = [], []
    for j in range(len(scam_temps)):
        for k in range(len(scam_srs)):
            lnZ = np.log(scam_srs[k]) + Q / (R * T_K_arr[j])
            lnsinh = np.log(np.sinh(alpha * sigma_matrix[j, k]))
            lnZ_all.append(lnZ)
            lnsinh_all.append(lnsinh)
    coeffs = np.polyfit(lnsinh_all, lnZ_all, 1)
    lnA = coeffs[1]
    
    results['strain'].append(eps)
    results['alpha'].append(alpha)
    results['n'].append(n_val)
    results['Q'].append(Q)
    results['lnA'].append(lnA)

# Convert to numpy arrays
for key in results:
    results[key] = np.array(results[key])

print(f"\n{'ε':>8s}  {'α (MPa⁻¹)':>12s}  {'n':>10s}  {'Q (J/mol)':>12s}  {'ln(A)':>10s}")
print("-" * 60)
for i in range(len(results['strain'])):
    print(f"{results['strain'][i]:8.4f}  {results['alpha'][i]:12.6f}  "
          f"{results['n'][i]:10.4f}  {results['Q'][i]:12.0f}  "
          f"{results['lnA'][i]:10.4f}")


# SECTION 3: 6TH-ORDER POLYNOMIAL FITTING

print("\n" + "=" * 70)
print("SECTION 3: 6th-order polynomial fitting for each parameter")
print("=" * 70)

poly_coeffs = {}
param_names = ['alpha', 'n', 'Q', 'lnA']
param_labels = ['α (MPa⁻¹)', 'n', 'Q (J/mol)', 'ln(A)']

for param, label in zip(param_names, param_labels):
    coeffs = np.polyfit(results['strain'], results[param], POLY_ORDER)
    poly_coeffs[param] = coeffs
    
    fitted = np.polyval(coeffs, results['strain'])
    ss_res = np.sum((results[param] - fitted) ** 2)
    ss_tot = np.sum((results[param] - np.mean(results[param])) ** 2)
    r2 = 1 - ss_res / ss_tot
    
    print(f"\n{'─' * 60}")
    print(f"  {label}(ε)  —  R² = {r2:.6f}")
    print(f"{'─' * 60}")
    print(f"  {label}(ε) =")
    for j, c in enumerate(coeffs):
        power = POLY_ORDER - j
        sign = "+" if c >= 0 else "-"
        abs_c = abs(c)
        if power == 0:
            print(f"    {sign} {abs_c:.10e}")
        elif power == 1:
            print(f"    {sign} {abs_c:.10e} · ε")
        else:
            print(f"    {sign} {abs_c:.10e} · ε^{power}")


# SECTION 4: SCAM PREDICTION FUNCTION


def scam_predict(eps, T_K, sr):
    """Predict true stress using SCAM polynomial Arrhenius model."""
    alpha = np.polyval(poly_coeffs['alpha'], eps)
    n     = np.polyval(poly_coeffs['n'], eps)
    Q     = np.polyval(poly_coeffs['Q'], eps)
    lnA   = np.polyval(poly_coeffs['lnA'], eps)
    Z     = sr * np.exp(Q / (R * T_K))
    sigma = (1.0 / alpha) * np.arcsinh((Z / np.exp(lnA)) ** (1.0 / n))
    return sigma


# SECTION 5: ERROR METRICS

print("\n" + "=" * 70)
print("SECTION 5: SCAM Prediction Error Metrics")
print("=" * 70)

all_temps = [25, 150, 250, 300, 350, 400, 450]
srs = [0.001, 0.01, 0.1]

# Collect per-condition metrics
metrics_rows = []
all_exp_15, all_pred_15 = [], []
all_exp_21, all_pred_21 = [], []

print(f"\n{'Condition':>15s}  {'R²':>8s}  {'RMSE (MPa)':>10s}  {'AARE (%)':>8s}")
print("-" * 50)

for T in all_temps:
    for sr in srs:
        T_K = T + 273.15
        sn = get_sheet_name(T, sr)
        d = all_data[sn]
        
        # Interpolate at common points within SCAM strain range
        eps_lo = max(strain_levels[0], d['eps_true'].min())
        eps_hi = min(strain_levels[-1], d['eps_true'].max())
        if eps_lo >= eps_hi:
            continue
        eps_common = np.linspace(eps_lo, eps_hi, 50)
        
        e = np.interp(eps_common, d['eps_true'], d['sigma_true'])
        p = scam_predict(eps_common, T_K, sr)
        
        valid = (e > 0) & np.isfinite(p) & (p > 0)
        e, p = e[valid], p[valid]
        
        if len(e) < 3:
            continue
        
        r2_i  = 1 - np.sum((e - p)**2) / np.sum((e - np.mean(e))**2)
        rmse_i = np.sqrt(np.mean((e - p)**2))
        aare_i = np.mean(np.abs((e - p) / e)) * 100
        
        label = "FIT" if T in scam_temps else "EXT"
        print(f"{sn:>15s}  {r2_i:8.4f}  {rmse_i:10.2f}  {aare_i:8.2f}  [{label}]")
        metrics_rows.append({
            'condition': sn, 'T_C': T, 'sr': sr,
            'R2': r2_i, 'RMSE': rmse_i, 'AARE': aare_i, 'type': label
        })
        
        all_exp_21.extend(e)
        all_pred_21.extend(p)
        if T in scam_temps:
            all_exp_15.extend(e)
            all_pred_15.extend(p)

# Overall metrics
all_exp_15, all_pred_15 = np.array(all_exp_15), np.array(all_pred_15)
all_exp_21, all_pred_21 = np.array(all_exp_21), np.array(all_pred_21)

for label, exp_arr, pred_arr in [("15 FIT conditions (250-450°C)", all_exp_15, all_pred_15),
                                   ("21 ALL conditions", all_exp_21, all_pred_21)]:
    r2   = 1 - np.sum((exp_arr - pred_arr)**2) / np.sum((exp_arr - np.mean(exp_arr))**2)
    rmse = np.sqrt(np.mean((exp_arr - pred_arr)**2))
    aare = np.mean(np.abs((exp_arr - pred_arr) / exp_arr)) * 100
    print(f"\n  OVERALL ({label}):")
    print(f"    R²   = {r2:.6f}")
    print(f"    RMSE = {rmse:.4f} MPa")
    print(f"    AARE = {aare:.2f}%")


# SECTION 6: FIGURES

print("\n" + "=" * 70)
print("SECTION 6: Generating figures")
print("=" * 70)


# FIGURE 1: 4 parameters vs strain + polynomial fit

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
eps_fine = np.linspace(strain_levels[0], strain_levels[-1], 300)

plot_data = [
    ('α (MPa⁻¹)', results['alpha'], poly_coeffs['alpha']),
    ('n',          results['n'],     poly_coeffs['n']),
    ('Q (J/mol)',  results['Q'],     poly_coeffs['Q']),
    ('ln(A)',      results['lnA'],   poly_coeffs['lnA']),
]

for ax, (label, vals, poly) in zip(axes.flat, plot_data):
    fitted_fine = np.polyval(poly, eps_fine)
    r2 = 1 - np.sum((vals - np.polyval(poly, results['strain']))**2) / \
             np.sum((vals - np.mean(vals))**2)
    
    ax.scatter(results['strain'], vals, c='red', s=60, zorder=5,
               edgecolors='darkred', linewidths=0.5, label='Calculated values')
    ax.plot(eps_fine, fitted_fine, 'b-', linewidth=2.5,
            label=f'6th-order polynomial (R²={r2:.4f})')
    ax.set_xlabel('True Strain ε', fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f'{label} vs True Strain', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('SCAM: Arrhenius Parameters as Functions of Strain\n'
             '(Fitted from 15 conditions: 250–450°C, 3 strain rates)',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(f'{OUTPUT_DIR}fig1_scam_parameters_vs_strain.png', dpi=200, bbox_inches='tight')
print("  Saved: fig1_scam_parameters_vs_strain.png")
plt.show()


# FIGURE 2: Predictions by TEMPERATURE (7 subplots)

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes_flat = list(axes.flat)

colors_sr = {0.001: '#1f77b4', 0.01: '#ff7f0e', 0.1: '#2ca02c'}
markers_sr = {0.001: 'o', 0.01: 's', 0.1: '^'}

for idx, T in enumerate(all_temps):
    ax = axes_flat[idx]
    T_K = T + 273.15
    
    for sr in srs:
        sn = get_sheet_name(T, sr)
        d = all_data[sn]
        
        # Experimental (thin line + scatter)
        step = max(1, len(d['eps_true']) // 30)
        ax.scatter(d['eps_true'][::step], d['sigma_true'][::step],
                   c=colors_sr[sr], marker=markers_sr[sr], s=20, alpha=0.7,
                   label=f'Exp {sr} s⁻¹')
        
        # SCAM prediction
        eps_lo = max(strain_levels[0], d['eps_true'].min())
        eps_hi = min(strain_levels[-1], d['eps_true'].max())
        if eps_lo < eps_hi:
            eps_pred = np.linspace(eps_lo, eps_hi, 150)
            sigma_pred = scam_predict(eps_pred, T_K, sr)
            ax.plot(eps_pred, sigma_pred, color=colors_sr[sr],
                    linewidth=2.5, linestyle='--', label=f'SCAM {sr} s⁻¹')
    
    title = f'T = RT (25°C)' if T == 25 else f'T = {T}°C'
    fit_label = '' if T in scam_temps else ' [EXTRAPOLATED]'
    ax.set_title(f'{title}{fit_label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('True Strain', fontsize=10)
    ax.set_ylabel('True Stress (MPa)', fontsize=10)
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

for idx in range(len(all_temps), 9):
    axes_flat[idx].axis('off')

plt.suptitle('SCAM Predictions vs Experimental — By Temperature\n'
             '(scatter = experimental, dashed = SCAM prediction)',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(f'{OUTPUT_DIR}fig2_scam_predictions_by_temperature.png', dpi=200, bbox_inches='tight')
print("  Saved: fig2_scam_predictions_by_temperature.png")
plt.show()


# FIGURE 3: Predictions by STRAIN RATE (3 subplots)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

colors_T = {
    25: '#d62728', 150: '#ff7f0e', 250: '#2ca02c', 300: '#1f77b4',
    350: '#9467bd', 400: '#8c564b', 450: '#e377c2'
}
markers_T = {25: 'o', 150: 's', 250: '^', 300: 'D', 350: 'v', 400: 'p', 450: '*'}

for ax, sr in zip(axes, srs):
    for T in all_temps:
        T_K = T + 273.15
        sn = get_sheet_name(T, sr)
        d = all_data[sn]
        
        step = max(1, len(d['eps_true']) // 20)
        T_label = 'RT' if T == 25 else f'{T}°C'
        ax.scatter(d['eps_true'][::step], d['sigma_true'][::step],
                   c=colors_T[T], marker=markers_T[T], s=25, alpha=0.7,
                   label=f'{T_label} exp')
        
        eps_lo = max(strain_levels[0], d['eps_true'].min())
        eps_hi = min(strain_levels[-1], d['eps_true'].max())
        if eps_lo < eps_hi:
            eps_pred = np.linspace(eps_lo, eps_hi, 150)
            sigma_pred = scam_predict(eps_pred, T_K, sr)
            ax.plot(eps_pred, sigma_pred, color=colors_T[T],
                    linewidth=2, linestyle='--')
    
    ax.set_title(f'ε̇ = {sr} s⁻¹', fontsize=14, fontweight='bold')
    ax.set_xlabel('True Strain', fontsize=12)
    ax.set_ylabel('True Stress (MPa)', fontsize=12)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

plt.suptitle('SCAM Predictions vs Experimental — By Strain Rate',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(f'{OUTPUT_DIR}fig3_scam_predictions_by_strain_rate.png', dpi=200, bbox_inches='tight')
print("  Saved: fig3_scam_predictions_by_strain_rate.png")
plt.show()


# FIGURE 4: Scatter plot — Predicted vs Experimental (15 conditions)

r2_15  = 1 - np.sum((all_exp_15 - all_pred_15)**2) / np.sum((all_exp_15 - np.mean(all_exp_15))**2)
rmse_15 = np.sqrt(np.mean((all_exp_15 - all_pred_15)**2))
aare_15 = np.mean(np.abs((all_exp_15 - all_pred_15) / all_exp_15)) * 100

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(all_exp_15, all_pred_15, s=15, alpha=0.5, c='steelblue', edgecolors='none')
lim = max(all_exp_15.max(), all_pred_15.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r-', linewidth=2, label='Perfect prediction (y=x)')
ax.plot([0, lim], [0, lim * 0.9], 'r--', linewidth=1, alpha=0.4, label='±10%')
ax.plot([0, lim], [0, lim * 1.1], 'r--', linewidth=1, alpha=0.4)
ax.set_xlabel('Experimental True Stress σ (MPa)', fontsize=13)
ax.set_ylabel('SCAM Predicted True Stress σ (MPa)', fontsize=13)
ax.set_title(f'SCAM: Predicted vs Experimental (15 fitting conditions)\n'
             f'R² = {r2_15:.4f}  |  RMSE = {rmse_15:.2f} MPa  |  AARE = {aare_15:.2f}%',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}fig4_scam_scatter_15conditions.png', dpi=200, bbox_inches='tight')
print("  Saved: fig4_scam_scatter_15conditions.png")
plt.show()


# FIGURE 5: Scatter plot — Predicted vs Experimental (ALL 21 conditions)

r2_21  = 1 - np.sum((all_exp_21 - all_pred_21)**2) / np.sum((all_exp_21 - np.mean(all_exp_21))**2)
rmse_21 = np.sqrt(np.mean((all_exp_21 - all_pred_21)**2))
aare_21 = np.mean(np.abs((all_exp_21 - all_pred_21) / all_exp_21)) * 100

fig, ax = plt.subplots(figsize=(8, 8))

# Color-code by whether it's a fitting condition or extrapolation
for T in all_temps:
    for sr in srs:
        T_K = T + 273.15
        sn = get_sheet_name(T, sr)
        d = all_data[sn]
        eps_lo = max(strain_levels[0], d['eps_true'].min())
        eps_hi = min(strain_levels[-1], d['eps_true'].max())
        if eps_lo >= eps_hi:
            continue
        eps_common = np.linspace(eps_lo, eps_hi, 50)
        e = np.interp(eps_common, d['eps_true'], d['sigma_true'])
        p = scam_predict(eps_common, T_K, sr)
        valid = (e > 0) & np.isfinite(p) & (p > 0)
        color = 'steelblue' if T in scam_temps else 'orangered'
        ax.scatter(e[valid], p[valid], s=12, alpha=0.5, c=color, edgecolors='none')

lim = max(all_exp_21.max(), all_pred_21.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r-', linewidth=2, label='Perfect prediction')
ax.scatter([], [], c='steelblue', s=30, label='250–450°C (fitting)')
ax.scatter([], [], c='orangered', s=30, label='RT & 150°C (extrapolation)')
ax.set_xlabel('Experimental True Stress σ (MPa)', fontsize=13)
ax.set_ylabel('SCAM Predicted True Stress σ (MPa)', fontsize=13)
ax.set_title(f'SCAM: All 21 Conditions\n'
             f'R² = {r2_21:.4f}  |  RMSE = {rmse_21:.2f} MPa  |  AARE = {aare_21:.2f}%',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}fig5_scam_scatter_21conditions.png', dpi=200, bbox_inches='tight')
print("  Saved: fig5_scam_scatter_21conditions.png")
plt.show()


# SECTION 7: POLYNOMIAL EQUATIONS — CLEAN PRINTOUT

print("\n" + "=" * 70)
print("SECTION 7: POLYNOMIAL EQUATIONS (copy-paste ready for paper)")
print("=" * 70)

for param, label in zip(param_names, param_labels):
    coeffs = poly_coeffs[param]
    fitted = np.polyval(coeffs, results['strain'])
    r2 = 1 - np.sum((results[param] - fitted)**2) / \
             np.sum((results[param] - np.mean(results[param]))**2)
    
    print(f"\n{label}(ε) = ", end="")
    terms = []
    for j, c in enumerate(coeffs):
        power = POLY_ORDER - j
        if power == 0:
            terms.append(f"({c:+.6e})")
        elif power == 1:
            terms.append(f"({c:+.6e})ε")
        else:
            terms.append(f"({c:+.6e})ε^{power}")
    print(" + ".join(terms))
    print(f"  R² = {r2:.6f}")

# Also print as a Python dict for direct use in PGNN code
print("\n\n# ═══ Python dict (copy into PGNN code for physics loss) ═══")
print("SCAM_POLY_COEFFS = {")
for param in param_names:
    c = poly_coeffs[param]
    print(f"    '{param}': np.array([{', '.join(f'{x:.12e}' for x in c)}]),")
print("}")

print("\n\n# ═══ Usage example ═══")
print("""
def scam_sigma(eps, T_K, sr, R=8.314):
    alpha = np.polyval(SCAM_POLY_COEFFS['alpha'], eps)
    n     = np.polyval(SCAM_POLY_COEFFS['n'], eps)
    Q     = np.polyval(SCAM_POLY_COEFFS['Q'], eps)
    lnA   = np.polyval(SCAM_POLY_COEFFS['lnA'], eps)
    Z     = sr * np.exp(Q / (R * T_K))
    sigma = (1.0 / alpha) * np.arcsinh((Z / np.exp(lnA)) ** (1.0 / n))
    return sigma
""")

print("=" * 70)
print("DONE — All figures saved to output directory.")
print("=" * 70)